# RSIAT on Kaggle

Notebook này chạy RSIAT trên Kaggle GPU. Mã nguồn được clone vào `/kaggle/working`, dữ liệu đầu vào đọc từ `/kaggle/input` (nếu đã attach Kaggle Dataset), còn log và checkpoint được ghi vào `/kaggle/working/RSIAT_outputs`. Bật Internet trong Kaggle nếu cần tải CIFAR-100 hoặc trọng số ViT pretrained.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/PhThuan-tech/RSIAT.git'
BRANCH = 'main'
PROJECT_DIR = Path('/kaggle/working/RSIAT')
OUTPUT_ROOT = Path('/kaggle/working/RSIAT_outputs')
DATA_ROOT = Path('/kaggle/working/RSIAT_data/datasets')
KAGGLE_INPUT_ROOT = Path('/kaggle/input')

# Set this to the mounted Kaggle Dataset directory when input auto-detection is ambiguous.
DATASET_INPUT_DIR = None

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

# The training code reads RSIAT_DATA_ROOT, so attached datasets can stay read-only.
os.environ['RSIAT_DATA_ROOT'] = str(DATA_ROOT)
os.chdir(PROJECT_DIR)

if DATASET_INPUT_DIR is None and KAGGLE_INPUT_ROOT.exists():
    candidates = [path for path in KAGGLE_INPUT_ROOT.iterdir() if path.is_dir()]
else:
    candidates = [Path(DATASET_INPUT_DIR)] if DATASET_INPUT_DIR else []

source_root = None
for candidate in candidates:
    if (candidate / 'cifar-100-python').exists() or any(
        (candidate / name).exists()
        for name in ('imagenet-r', 'imagenet-a', 'cub', 'vtab', 'omnibenchmark')
    ):
        source_root = candidate
        break
    if candidate.name == 'cifar-100-python':
        source_root = candidate.parent
        break

if source_root is not None:
    for item in source_root.iterdir():
        destination = DATA_ROOT / item.name
        if destination.exists() or destination.is_symlink():
            continue
        if item.is_dir():
            destination.symlink_to(item, target_is_directory=True)
        else:
            shutil.copy2(item, destination)
    print('Attached dataset:', source_root)
else:
    print('No attached dataset detected; CIFAR-100 download requires Kaggle Internet access.')

print('Project:', Path.cwd())
print('Data:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
!nvidia-smi
import torch, torchvision
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('torchvision:', torchvision.__version__)

# Kaggle images usually include torch/torchvision; install only project dependencies.
%pip install -q --upgrade-strategy only-if-needed -r requirements-colab.txt
import timm
print('timm:', timm.__version__)
!python -m py_compile main.py trainer.py models/base.py models/RSIAT_adapter.py data/data.py

In [ ]:
import json

# CIFAR-100 is downloaded to the writable Kaggle working directory when absent.
from torchvision.datasets import CIFAR100

cifar_train = CIFAR100(root=str(DATA_ROOT), train=True, download=True)
cifar_test = CIFAR100(root=str(DATA_ROOT), train=False, download=True)


def make_kaggle_config(source, destination, seed=1993, resume=False, max_tasks_per_run=None):
    config = json.loads((PROJECT_DIR / source).read_text())
    config.update({
        'seed': [seed],
        'resume': resume,
        'output_root': str(OUTPUT_ROOT),
        'device': ['0'] if torch.cuda.is_available() else [-1],
        'num_workers': 2,
        'stats_num_workers': 2,
        'persistent_workers': True,
    })
    if max_tasks_per_run is not None:
        config['max_tasks_per_run'] = max_tasks_per_run
    Path(destination).write_text(json.dumps(config, indent=2))

make_kaggle_config('exps/adapter_cifar224_smoke.json', '/kaggle/working/rsiat_smoke.json', max_tasks_per_run=1)
make_kaggle_config('exps/adapter_cifar224.json', '/kaggle/working/rsiat_full.json', resume=True)
print('CIFAR train/test:', len(cifar_train), len(cifar_test))

## Smoke test

Smoke test chạy task đầu tiên với 1 epoch, kiểm tra data, pretrained ViT, train, evaluation và checkpoint trên Kaggle.

In [ ]:
!python main.py --config /kaggle/working/rsiat_smoke.json

## Full CIFAR-100 experiment

Config full bật `resume`: checkpoint và log nằm trong `/kaggle/working/RSIAT_outputs`. Nếu muốn giữ kết quả sau khi phiên Kaggle kết thúc, hãy bật Save Version hoặc xuất thư mục output thành Kaggle Dataset.

In [ ]:
# Bỏ comment sau khi smoke test thành công.
# !python main.py --config /kaggle/working/rsiat_full.json

In [ ]:
print('Recent logs:')
for path in sorted((OUTPUT_ROOT / 'logs' / 'adapter').rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)
print('Recent checkpoints:')
for path in sorted((OUTPUT_ROOT / 'ckpt').rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)